In [ ]:
import tensorflow as tf
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt


df_temperatura = pd.read_csv('celsius_a_fahrenheit.csv')

print("--- Primeros 5 registros de la base de datos ---")
print(df_temperatura.head())
print("-" * 50)

plt.figure(figsize=(8, 6))
sns.scatterplot(x='Celsius', y='Fahrenheit', data=df_temperatura)
plt.title('Correlación entre Celsius y Fahrenheit')
plt.grid(True)
plt.show()


X_train = df_temperatura['Celsius'].values
y_train = df_temperatura['Fahrenheit'].values


modelo_simple = tf.keras.Sequential([
    tf.keras.layers.Dense(units=1, input_shape=[1])
])


oculta1 = tf.keras.layers.Dense(units=3, input_shape=[1])
oculta2 = tf.keras.layers.Dense(units=3)
salida = tf.keras.layers.Dense(units=1)

modelo_multicapa = tf.keras.Sequential([oculta1, oculta2, salida])


modelo_simple.compile(
    optimizer=tf.keras.optimizers.Adam(0.1), 
    loss='mean_squared_error'
)

modelo_multicapa.compile(
    optimizer=tf.keras.optimizers.Adam(0.1), 
    loss='mean_squared_error'
)


print("Entrenando el modelo de 1 capa...")
historial_simple = modelo_simple.fit(X_train, y_train, epochs=300, verbose=False)

print("Entrenando el modelo de 3 capas...")
historial_multicapa = modelo_multicapa.fit(X_train, y_train, epochs=300, verbose=False)


plt.figure(figsize=(10, 6))
plt.plot(historial_simple.history['loss'], label='Pérdida - 1 Capa', linestyle='--')
plt.plot(historial_multicapa.history['loss'], label='Pérdida - 3 Capas')
plt.title('Progreso de Pérdida durante el Entrenamiento (MSE)')
plt.xlabel('Epoch')
plt.ylabel('Training Loss')
plt.legend()
plt.grid(True)
plt.show()


print("\n--- Pesos Internos del Modelo Multicapa ---")
print("\nCapa Oculta 1:")
print(oculta1.get_weights())

print("\nCapa Oculta 2:")
print(oculta2.get_weights())

print("\nCapa de Salida:")
print(salida.get_weights())


Temp_C = np.array([[100.0]])
prediccion_simple = modelo_simple.predict(Temp_C, verbose=False)
prediccion_multicapa = modelo_multicapa.predict(Temp_C, verbose=False)

print("\n--- Resultados de Predicción para 100°C ---")
print(f"Temperatura Real (Fórmula Matemática): 212.00 °F")
print(f"Predicción Modelo 1 Capa: {prediccion_simple[0][0]:.2f} °F")
print(f"Predicción Modelo 3 Capas: {prediccion_multicapa[0][0]:.2f} °F")

## 1. Optimizadores (Adam, SGD y RMSProp)
### El optimizador Adam, configurado en tu código, adapta la tasa de aprendizaje combinando las heurísticas de Momentum y RMSProp, lo que le permite converger rápidamente al mínimo error. SGD (Descenso de Gradiente Estocástico) actualiza los pesos basándose únicamente en el gradiente negativo actual; si lo utilizaras, el entrenamiento se ralentizaría drásticamente y podría no converger sin ajustar la tasa de aprendizaje manualmente. Por otro lado, RMSProp adapta la tasa de aprendizaje dividiendo el gradiente por una media móvil de magnitudes recientes; si lo implementaras en este modelo lineal, obtendrías una velocidad y resultados de convergencia casi idénticos a los de Adam.

## 2. Función de Pérdida (MSE) y Alternativas
### La función 'mean_squared_error' es fundamental porque cuantifica la exactitud del modelo calculando el promedio de los errores al cuadrado entre las predicciones y los valores reales , lo cual le indica matemáticamente a la red en qué dirección debe ajustar los pesos para minimizar el fallo. Una alternativa directa es el Error Absoluto Medio (MAE), que promedia las diferencias absolutas sin elevarlas al cuadrado; su implementación alteraría la curva de aprendizaje durante el entrenamiento, haciéndola descender de forma lineal en lugar de exponencial, aunque el resultado de la predicción final sería idéntico ya que los datos de conversión de temperatura no contienen ruido estadístico.

## 3. Épocas suficientes para el entrenamiento
### Al observar el comportamiento de la curva de pérdida generada en las gráficas, el error cae bruscamente y se estabiliza cerca de cero en las primeras etapas del proceso. Por lo tanto, entre 50 y 100 épocas fueron matemáticamente suficientes para que el modelo aprendiera la relación determinista entre los grados; ejecutar las 300 o 500 épocas iniciales resulta en un exceso de computación sin mejora en la predicción.

## 4. Importancia de las Epochs
### Una época (epoch) es la métrica de iteración que indica que el algoritmo ha procesado la totalidad del conjunto de los datos de entrenamiento exactamente una vez. Son indispensables porque la optimización por descenso de gradiente es estrictamente un proceso numérico iterativo; una red neuronal no puede calcular la fórmula matemática subyacente de una sola pasada, requiriendo múltiples repeticiones continuas sobre los datos para ir recalibrando paulatinamente sus pesos hasta minimizar el error residual de la ecuación.

## 5. Funciones de activación y el impacto de ReLU
### La capa densa configurada en el modelo no declara explícitamente una función de activación, por lo que el framework utiliza por defecto una función de activación Lineal ($f(x) = x$), lo cual es la práctica correcta para una regresión continua. Si cambias esta capa utilizando activation='relu', se aplicará la lógica matemática $\max(0, x)$, lo cual destruirá el modelo al predecir temperaturas frías; por ejemplo, si se introducen -30 °C, la fórmula interna calcularía -22 °F, pero la función ReLU truncaría instantáneamente ese valor negativo y mostraría como resultado final 0 °F.